# 06 · Robustness
Every candidate effect faces the same battery: (a) excluding eco rounds,
(b) taxonomy k ± 1, (c) per-map splits, (d) a permutation test that destroys
player-level consistency while keeping each round's role mix intact.
All variant computations use `save=False` so the canonical parquet artifacts
written by notebooks 04–05 are never overwritten.

In [ ]:
# --- bootstrap (identical in every notebook) ---------------------------------
from google.colab import drive
drive.mount('/content/drive')
%pip -q install demoparser2 minisom

import sys
sys.path.insert(0, '/content/drive/MyDrive/cs2-btp/code')

from cs2btp import (config as cfg, manifest as mf, parsing, qc,
                    features as ft, roles, consistency as cons,
                    outcome as out, viz)
cfg.ensure_dirs()
print('pipeline ready | drive root =', cfg.DRIVE_ROOT)

In [ ]:
import pandas as pd
labeled = roles.load_labeled()
feats = ft.load_all()
team_round = pd.read_parquet(cfg.ANALYSIS/'team_round_consistency.parquet')
ds = out.build_round_dataset(team_round, save=False)
base_model, _ = out.fit_main_logit(ds)
print('baseline cons_diff beta = %.3f (p=%.4f)' %
      (base_model.params['cons_diff'], base_model.pvalues['cons_diff']))

## (a) Exclude eco rounds

In [ ]:
no_eco = labeled[labeled.buy_type != 'eco']
tr = cons.team_round_table(cons.rolling(no_eco, save=False), save=False)
m, d = out.fit_main_logit(out.build_round_dataset(tr, save=False))
print('no-eco: beta = %.3f  p = %.4f  n = %d' %
      (m.params['cons_diff'], m.pvalues['cons_diff'], len(d)))

## (b) k ± 1 roles

In [ ]:
K0 = {s: int(labeled.loc[labeled.side==s,'role_id'].nunique()) for s in ('T','CT')}
for dk in (-1, +1):
    kk = {s: max(3, K0[s]+dk) for s in K0}
    lab_k, _ = roles.discover_roles(feats, kk, save=False)
    tr = cons.team_round_table(cons.rolling(lab_k, save=False), save=False)
    m, d = out.fit_main_logit(out.build_round_dataset(tr, save=False))
    print(f'k{dk:+d}: beta = {m.params["cons_diff"]: .3f}  '
          f'p = {m.pvalues["cons_diff"]:.4f}  n = {len(d)}')

## (c) Per-map coefficients (noisy — direction is what matters)

In [ ]:
for mp, g in ds.dropna(subset=['cons_diff']).groupby('map_name'):
    if len(g) < 80 or g.demo_id.nunique() < 3:
        continue
    try:
        m, d = out.fit_main_logit(g)
        print(f'{mp:10s} beta = {m.params["cons_diff"]: .3f}  n = {len(d)}')
    except Exception as e:
        print(mp, 'skipped:', e)

## (d) Permutation test (~10–20 min for 100 permutations)

In [ ]:
res = out.permutation_test(labeled, n_perm=100)
print(res)
import json
with open(cfg.ANALYSIS/'permutation_test.json','w') as fh:
    json.dump(res, fh, indent=2)

## (e) Robustness battery for the weakest-link measure (`cons_min_diff`)
The one prospective coefficient that was individually significant in
notebook 05 gets the identical battery (~30–40 min, mostly the permutation).
Decision rule (fixed in advance): report as robust only if the point estimate
keeps its sign and rough size across no-eco and k ± 1 AND the permutation p
is small; a sign flip under subsetting marks it specification-dependent.

In [ ]:
import numpy as np, json

def fit_min(ds_):
    m, d_ = out.fit_main_logit(ds_, cons_var='cons_min_diff')
    return m.params['cons_min_diff'], m.pvalues['cons_min_diff'], len(d_)

b, p, n = fit_min(ds); print(f'baseline : beta={b: .3f}  p={p:.4f}  n={n}')

tr_ne = cons.team_round_table(cons.rolling(labeled[labeled.buy_type != 'eco'],
                                           save=False), save=False)
b, p, n = fit_min(out.build_round_dataset(tr_ne, save=False))
print(f'no-eco   : beta={b: .3f}  p={p:.4f}  n={n}')

for dk in (-1, +1):
    kk = {s: max(3, K0[s]+dk) for s in K0}
    lab_k, _ = roles.discover_roles(feats, kk, save=False)
    tr_k = cons.team_round_table(cons.rolling(lab_k, save=False), save=False)
    b, p, n = fit_min(out.build_round_dataset(tr_k, save=False))
    print(f'k{dk:+d}      : beta={b: .3f}  p={p:.4f}  n={n}')

res_min = out.permutation_test(labeled, n_perm=100, cons_var='cons_min_diff')
print('permutation:', res_min)
with open(cfg.ANALYSIS/'robustness_cons_min.json','w') as fh:
    json.dump(res_min, fh, indent=2)